# Phân loại ảnh sản phẩm thời trang bằng CNN — Xử lý dữ liệu (dùng chung cho cả nhóm)

**Dataset:** Fashion Product Images (Small) — https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small

**Các bước trong notebook này**
1. Đọc `styles.csv` (sửa 22 dòng bị hỏng), ghép ảnh với nhãn qua `id`.
2. Chỉ giữ 10 nhãn `articleType`.
3. Kiểm tra ảnh tồn tại, đọc được, đúng kích thước gốc 60×80 (rộng×cao).
4. Chia train / validation / test (70/15/15) theo từng nhãn, lưu `split.csv`.
5. Đọc ảnh RGB, resize thành **96×72×3** (cao×rộng, giữ tỉ lệ 3:4 của ảnh gốc), lưu dạng `uint8` vào `fashion_96x72.npz`.
6. Tính class weights, tạo lớp data augmentation (chỉ dùng khi train).

**Quy ước chung cho cả ba mô hình**
- Đầu vào: ảnh `uint8` kích thước `(96, 72, 3)`. **Không tự đổi kích thước.**
- Chuẩn hóa nằm *bên trong* từng mô hình: `Rescaling(1/255)` cho Simple CNN và Complex CNN, `preprocess_input` cho MobileNetV2.
- Thứ tự lớp (`class_names`) sắp xếp theo bảng chữ cái, dùng chung để Confusion Matrix không bị lệch nhãn.
- Chỉ dùng tập test **một lần** ở cuối để so sánh mô hình; dùng validation để chỉnh mô hình.

In [ ]:
import csv
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import layers
import kagglehub

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# Kích thước đầu vào của mô hình (cao x rộng). Ảnh gốc là 80 x 60 nên giữ tỉ lệ 3:4.
IMG_H, IMG_W = 96, 72

## 1. Tải dữ liệu và đọc `styles.csv`

In [ ]:
path = kagglehub.dataset_download("paramaggarwal/fashion-product-images-small")
print("Dữ liệu Kaggle đã sẵn sàng tại:", path)

In [ ]:
folder_data = Path(path)
print(list(folder_data.iterdir()))

In [ ]:
csv_path = folder_data / "styles.csv"
images_path = folder_data / "images"

print("styles.csv tồn tại:", csv_path.exists())
print("images tồn tại:", images_path.exists())

In [ ]:
fixed_csv = []
fixed_count = 0
short_rows = []

with open(csv_path, "r", encoding="utf-8-sig", newline="") as new_csv:
    line_row = csv.reader(new_csv)
    header = next(line_row)

    for line_number, row in enumerate(line_row, start=2):
        if len(row) > len(header):
            # Giữ nguyên 9 cột đầu, ghép phần dư vào tên sản phẩm
            row = row[:9] + [", ".join(row[9:])]
            fixed_count += 1

        elif len(row) < len(header):
            short_rows.append((line_number, row))
            continue

        fixed_csv.append(row)

data = pd.DataFrame(fixed_csv, columns=header)

print("Số dòng đã sửa:", fixed_count)
print("Số dòng thiếu cột:", len(short_rows))
print("Kích thước dữ liệu:", data.shape)
print("Số cột sai sau xử lý:", sum(len(row) != len(header) for row in fixed_csv))
display(data.head())

## 2. Ghép ảnh với nhãn và làm sạch

In [ ]:
def find_image(id):
    for ext in [".jpg", ".jpeg", ".png", ".webp"]:
        file_path = images_path / f"{int(id)}{ext}"
        if file_path.is_file():
            return str(file_path)
    return None

data["image_path"] = data["id"].apply(find_image)

missing_row = data["image_path"].isna().sum()
print("Số dòng không tìm thấy ảnh:", missing_row)

data = data.dropna(subset=["image_path"]).copy()

print("Số dòng còn lại:", len(data))
print("Số dòng thiếu ảnh sau xử lý:", data["image_path"].isna().sum())

In [ ]:
columns = data.select_dtypes(include=["object"]).columns
for column in columns:
    data[column] = data[column].str.strip()

print("ID bị thiếu:", data["id"].isna().sum())
print("ID trùng:", data["id"].duplicated().sum())
print("Nhãn articleType bị thiếu:", data["articleType"].isna().sum())

In [ ]:
print("Số nhãn trong dataset:", data["articleType"].nunique())
print(data["articleType"].value_counts().head(15))

## 3. Chọn 10 nhãn

In [ ]:
select_classes = [
    "Tshirts",
    "Shirts",
    "Tops",
    "Kurtas",
    "Jeans",
    "Casual Shoes",
    "Sports Shoes",
    "Heels",
    "Flip Flops",
    "Handbags",
]
data = data[data["articleType"].isin(select_classes)].copy()

print("Tổng số ảnh thuộc 10 nhãn:", len(data))
print("\nSố ảnh của từng nhãn:")
print(data["articleType"].value_counts().reindex(select_classes, fill_value=0))

## 4. Kiểm tra ảnh đọc được và đúng kích thước

Bước này đặt **trước** khi chia tập, để ảnh hỏng không lọt vào train/val/test.
Kích thước gốc kỳ vọng là `(60, 80)` (rộng, cao).

In [ ]:
def check_image(p):
    try:
        with Image.open(p) as im:
            im.verify()
        with Image.open(p) as im:  # mở lại vì verify() làm đối tượng không dùng tiếp được
            return im.size          # (rộng, cao)
    except Exception:
        return None

data["size"] = [check_image(p) for p in tqdm(data["image_path"])]

print("Ảnh hỏng/không đọc được:", data["size"].isna().sum())
print("\nKích thước ảnh gốc (rộng, cao):")
print(data["size"].value_counts())

data = data.dropna(subset=["size"]).drop(columns="size").reset_index(drop=True)
print("\nSố ảnh còn lại:", len(data))

## 5. Chia train / validation / test và lưu `split.csv`

- Tỉ lệ 70 / 15 / 15, chia theo từng nhãn (`stratify`), `random_state` cố định.
- **Chia trước khi augment**, augmentation chỉ áp dụng cho tập train.
- `split.csv` giúp cả nhóm dùng *đúng cùng một* cách chia.

In [ ]:
train_data, remain_data = train_test_split(
    data, test_size=0.30, random_state=RANDOM_STATE, stratify=data["articleType"])
val_data, test_data = train_test_split(
    remain_data, test_size=0.50, random_state=RANDOM_STATE, stratify=remain_data["articleType"])

# Mã hóa nhãn: thứ tự theo bảng chữ cái, dùng chung cho cả nhóm
class_names = sorted(select_classes)
label_to_idx = {c: i for i, c in enumerate(class_names)}
NUM_CLASSES = len(class_names)

y_train = train_data["articleType"].map(label_to_idx).values
y_val = val_data["articleType"].map(label_to_idx).values
y_test = test_data["articleType"].map(label_to_idx).values

print("Train:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))
print("Tổng:", len(train_data) + len(val_data) + len(test_data))
print("Thứ tự lớp:", class_names)

split_df = pd.concat([
    train_data.assign(split="train"),
    val_data.assign(split="val"),
    test_data.assign(split="test"),
])[["id", "articleType", "split"]]
split_df.to_csv("split.csv", index=False)
print("\nĐã lưu split.csv:", split_df.shape)

## 6. Đọc ảnh, resize 96×72 và lưu cache

Ảnh được lưu dạng `uint8` (0–255). Việc chuẩn hóa để lại cho từng mô hình.

In [ ]:
def load_images(df):
    X = np.zeros((len(df), IMG_H, IMG_W, 3), dtype="uint8")
    for i, p in enumerate(tqdm(df["image_path"])):
        with Image.open(p) as im:
            # PIL.resize nhận (rộng, cao)
            X[i] = np.asarray(im.convert("RGB").resize((IMG_W, IMG_H), Image.BILINEAR))
    return X

X_train = load_images(train_data)
X_val = load_images(val_data)
X_test = load_images(test_data)

print("X_train:", X_train.shape, X_train.dtype)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)

In [ ]:
np.savez_compressed(
    "fashion_96x72.npz",
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    X_test=X_test, y_test=y_test,
    class_names=np.array(class_names),
)
print("Đã lưu fashion_96x72.npz")

# Nếu chạy trên Colab, nên copy lên Google Drive để không mất khi hết phiên:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp fashion_96x72.npz split.csv /content/drive/MyDrive/

## 7. Phân phối lớp và class weights

In [ ]:
dist = pd.DataFrame({
    "train": np.bincount(y_train, minlength=NUM_CLASSES),
    "val": np.bincount(y_val, minlength=NUM_CLASSES),
    "test": np.bincount(y_test, minlength=NUM_CLASSES),
}, index=class_names)
dist["total"] = dist.sum(axis=1)
display(dist)

dist["train"].sort_values(ascending=False).plot(
    kind="bar", figsize=(8, 4), title="Số ảnh mỗi lớp (tập train)")
plt.ylabel("Số ảnh")
plt.tight_layout()
plt.show()

# Lớp nhiều ảnh (Tshirts) hơn lớp ít ảnh (Jeans) gần 12 lần -> dùng class weights khi train
w = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_train)
class_weight = dict(enumerate(w))
print({class_names[k]: round(float(v), 2) for k, v in class_weight.items()})

## 8. Data augmentation (chỉ dùng khi train)

Các lớp ngẫu nhiên của Keras tự tắt khi `predict` / `evaluate`, nên validation và test không bị augment.
Không dùng lật dọc vì ảnh quần áo, giày và túi không bị lật ngược.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.1, value_range=(0, 255)),
], name="data_augmentation")

In [ ]:
# Hình minh họa cho báo cáo: ảnh gốc (hàng trên) và sau augmentation (hàng dưới)
fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(1.8 * NUM_CLASSES, 4.2))
for c in range(NUM_CLASSES):
    img = X_train[np.where(y_train == c)[0][0]]
    aug = data_augmentation(img[None].astype("float32"), training=True)[0].numpy()
    axes[0, c].imshow(img)
    axes[0, c].set_title(class_names[c], fontsize=8)
    axes[1, c].imshow(np.clip(aug, 0, 255).astype("uint8"))
    axes[0, c].axis("off")
    axes[1, c].axis("off")
fig.text(0.01, 0.72, "Gốc", rotation=90, va="center")
fig.text(0.01, 0.28, "Augment", rotation=90, va="center")
plt.tight_layout()
plt.show()

## 9. Cách dùng cho Người 2 và Người 3

**Cách 1 — nạp file đã xử lý (khuyên dùng):**
```python
d = np.load("fashion_96x72.npz")
X_train, y_train = d["X_train"], d["y_train"]
X_val,   y_val   = d["X_val"],   d["y_val"]
X_test,  y_test  = d["X_test"],  d["y_test"]
class_names = list(d["class_names"])
```

**Khung đầu vào của mô hình** (khác nhau ở bước chuẩn hóa):
```python
inputs = layers.Input(shape=(96, 72, 3))
x = data_augmentation(inputs)
x = layers.Rescaling(1/255)(x)   # Simple CNN, Complex CNN
# MobileNetV2: thay dòng trên bằng
# x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
```
Khi train: `model.fit(X_train, y_train, validation_data=(X_val, y_val), class_weight=class_weight, ...)`
(dùng `loss="sparse_categorical_crossentropy"` vì nhãn là số nguyên).

Khi tạo MobileNetV2 với ảnh không vuông, Keras có thể in cảnh báo về `input_shape`; trọng số ImageNet vẫn được nạp bình thường.